# 01 — Data Exploration
Explore Synthea/mCODE synthetic cancer patient data before loading into Azure SQL.

In [1]:
import json
import os
import pandas as pd

# Path to your data folders
mixed_path = "../data/raw/mixed-cancer"
breast_path = "../data/raw/breast-cancer"

# Get all patient JSON files
def get_patient_files(folder):
    files = [f for f in os.listdir(folder) 
             if f.endswith('.json') 
             and 'hospital' not in f.lower() 
             and 'practitioner' not in f.lower()]
    return files

mixed_files = get_patient_files(mixed_path)
breast_files = get_patient_files(breast_path)

print(f"Mixed cancer patients: {len(mixed_files)}")
print(f"Breast cancer patients: {len(breast_files)}")
print(f"Total patients: {len(mixed_files) + len(breast_files)}")

Mixed cancer patients: 2100
Breast cancer patients: 1113
Total patients: 3213


In [2]:
# Load and explore one patient FHIR file
import json

# Load first breast cancer patient
first_patient_file = os.path.join(breast_path, breast_files[0])

with open(first_patient_file, 'r') as f:
    patient_data = json.load(f)

# See what's inside
print(f"File: {breast_files[0]}")
print(f"Resource type: {patient_data['resourceType']}")
print(f"Total resources in file: {len(patient_data['entry'])}")
print()

# Show all resource types in this patient's record
resource_types = [entry['resource']['resourceType'] 
                  for entry in patient_data['entry']]
from collections import Counter
counts = Counter(resource_types)
print("Resource types found:")
for resource, count in sorted(counts.items()):
    print(f"  {resource}: {count}")

File: Katherine209_Reichert620_8c0d6e0a-2cbe-436d-0cfc-2e645de0c71a.json
Resource type: Bundle
Total resources in file: 289

Resource types found:
  CarePlan: 1
  CareTeam: 1
  Condition: 5
  DiagnosticReport: 26
  DocumentReference: 24
  Encounter: 24
  ImagingStudy: 1
  Immunization: 20
  MedicationRequest: 7
  Observation: 126
  Patient: 1
  Procedure: 52
  Provenance: 1


In [3]:
# Extract patient demographics from one file
patient_resource = next(
    entry['resource'] for entry in patient_data['entry']
    if entry['resource']['resourceType'] == 'Patient'
)

print(f"Patient ID: {patient_resource['id']}")
print(f"Gender: {patient_resource.get('gender', 'Unknown')}")
print(f"Birth Date: {patient_resource.get('birthDate', 'Unknown')}")

# Get name
name = patient_resource.get('name', [{}])[0]
given = ' '.join(name.get('given', []))
family = name.get('family', '')
print(f"Name: {given} {family}")

# Get conditions (diagnoses)
conditions = [
    entry['resource'] for entry in patient_data['entry']
    if entry['resource']['resourceType'] == 'Condition'
]

print(f"\nConditions ({len(conditions)}):")
for condition in conditions:
    code = condition.get('code', {})
    text = code.get('text', 'Unknown condition')
    print(f"  - {text}")

Patient ID: 8c0d6e0a-2cbe-436d-0cfc-2e645de0c71a
Gender: female
Birth Date: 1997-12-24
Name: Katherine209 Reichert620

Conditions (5):
  - Streptococcal sore throat (disorder)
  - Viral sinusitis (disorder)
  - Malignant neoplasm of breast (disorder)
  - Otitis media
  - Fracture of clavicle


In [4]:
# Extract cancer patients across ALL files
import json
import os
from tqdm import tqdm

cancer_keywords = [
    'malignant', 'carcinoma', 'neoplasm', 
    'cancer', 'tumor', 'lymphoma', 'leukemia'
]

def extract_cancer_patients(folder, dataset_name):
    results = []
    files = get_patient_files(folder)
    
    for filename in tqdm(files[:50], desc=f"Processing {dataset_name}"):
        filepath = os.path.join(folder, filename)
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        # Get patient demographics
        patient = next(
            (e['resource'] for e in data['entry'] 
             if e['resource']['resourceType'] == 'Patient'), None
        )
        if not patient:
            continue
            
        # Get cancer conditions only
        conditions = [
            e['resource'] for e in data['entry']
            if e['resource']['resourceType'] == 'Condition'
        ]
        
        cancer_conditions = [
            c for c in conditions
            if any(kw in c.get('code', {}).get('text', '').lower() 
                   for kw in cancer_keywords)
        ]
        
        if cancer_conditions:
            name = patient.get('name', [{}])[0]
            results.append({
                'patient_id': patient['id'],
                'gender': patient.get('gender'),
                'birth_date': patient.get('birthDate'),
                'cancer_type': cancer_conditions[0].get('code', {}).get('text'),
                'dataset': dataset_name
            })
    
    return results

# Process first 50 patients from each dataset
breast_patients = extract_cancer_patients(breast_path, "Breast Cancer")
mixed_patients = extract_cancer_patients(mixed_path, "Mixed Cancer")

# Combine into DataFrame
df = pd.DataFrame(breast_patients + mixed_patients)
print(f"\nTotal cancer patients found: {len(df)}")
print(f"\nCancer types:")
print(df['cancer_type'].value_counts())

Processing Mixed Cancer: 100%|██████████| 50/50 [00:00<00:00, 107.77it/s]


Total cancer patients found: 94

Cancer types:
cancer_type
Malignant neoplasm of breast (disorder)       45
Suspected lung cancer (situation)             12
Secondary malignant neoplasm of colon         10
Malignant tumor of colon                      10
Primary malignant neoplasm of colon            5
Overlapping malignant neoplasm of colon        5
Neoplasm of prostate                           4
Acute myeloid leukemia, disease (disorder)     3
Name: count, dtype: int64


In [5]:
# Azure SQL Schema — create tables for oncology patient data
schema_sql = """
-- 1. Patients table
CREATE TABLE Patients (
    patient_id NVARCHAR(100) PRIMARY KEY,
    first_name NVARCHAR(100),
    last_name NVARCHAR(100),
    gender NVARCHAR(20),
    birth_date DATE,
    race NVARCHAR(100),
    ethnicity NVARCHAR(100),
    state NVARCHAR(50),
    dataset_source NVARCHAR(50)
);

-- 2. Conditions table (cancer diagnosis)
CREATE TABLE Conditions (
    condition_id NVARCHAR(100) PRIMARY KEY,
    patient_id NVARCHAR(100),
    condition_code NVARCHAR(50),
    condition_text NVARCHAR(500),
    onset_date DATE,
    abatement_date DATE,
    is_cancer BIT,
    cancer_type NVARCHAR(100),
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);

-- 3. Treatments table (medications + procedures)
CREATE TABLE Treatments (
    treatment_id NVARCHAR(100) PRIMARY KEY,
    patient_id NVARCHAR(100),
    treatment_type NVARCHAR(50),
    treatment_code NVARCHAR(50),
    treatment_text NVARCHAR(500),
    start_date DATE,
    end_date DATE,
    status NVARCHAR(50),
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);

-- 4. Observations table (lab results, tumor markers)
CREATE TABLE Observations (
    observation_id NVARCHAR(100) PRIMARY KEY,
    patient_id NVARCHAR(100),
    observation_code NVARCHAR(50),
    observation_text NVARCHAR(500),
    observation_value NVARCHAR(200),
    observation_unit NVARCHAR(50),
    observation_date DATE,
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);

-- 5. Encounters table (hospital visits)
CREATE TABLE Encounters (
    encounter_id NVARCHAR(100) PRIMARY KEY,
    patient_id NVARCHAR(100),
    encounter_type NVARCHAR(100),
    encounter_date DATE,
    end_date DATE,
    reason_text NVARCHAR(500),
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);
"""

print("Schema designed successfully!")
print("Tables to create:")
print("  1. Patients")
print("  2. Conditions (cancer diagnoses)")
print("  3. Treatments (medications + procedures)")
print("  4. Observations (lab results, tumor markers)")
print("  5. Encounters (hospital visits)")

Schema designed successfully!
Tables to create:
  1. Patients
  2. Conditions (cancer diagnoses)
  3. Treatments (medications + procedures)
  4. Observations (lab results, tumor markers)
  5. Encounters (hospital visits)


In [6]:
import pyodbc
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv('../.env')

# Connection string
server = os.getenv('AZURE_SQL_SERVER')
database = os.getenv('AZURE_SQL_DATABASE')
connection_string = os.getenv('AZURE_SQL_CONNECTION_STRING')

print(f"Connecting to: {server}")
print(f"Database: {database}")

# Connect to Azure SQL
try:
    conn = pyodbc.connect(connection_string)
    cursor = conn.cursor()
    print("✅ Connected to Azure SQL successfully!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

ImportError: dlopen(/Users/bhargavi/Downloads/oncology-care-insights-agent/.venv/lib/python3.14/site-packages/pyodbc.cpython-314-darwin.so, 0x0002): Library not loaded: /opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib
  Referenced from: <D2ACAA22-1BC5-3553-8ED4-1A4758D72807> /Users/bhargavi/Downloads/oncology-care-insights-agent/.venv/lib/python3.14/site-packages/pyodbc.cpython-314-darwin.so
  Reason: tried: '/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file), '/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file)

In [11]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../.env')

server = os.getenv('AZURE_SQL_SERVER')
database = os.getenv('AZURE_SQL_DATABASE')
username = 'oncologyadmin'
password = os.getenv('AZURE_SQL_PASSWORD')

print(f"Connecting to: {server}")
print(f"Database: {database}")

# Connect
try:
    engine = create_engine(
        f"mssql+pymssql://{username}:{password}@{server}/{database}"
    )
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@VERSION"))
        version = result.fetchone()[0]
        print("✅ Connected successfully!")
        print(f"SQL Server: {version[:50]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

Connecting to: oncology-sql-server.database.windows.net
Database: oncology-patients
❌ Connection failed: (pymssql.exceptions.OperationalError) (20009, b'DB-Lib error message 20009, severity 9:\nUnable to connect: Adaptive Server is unavailable or does not exist (2026!@oncology-sql-server.database.windows.net)\n')
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [12]:
import pymssql
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../.env')

# Connect directly with pymssql
try:
    conn = pymssql.connect(
        server='oncology-sql-server.database.windows.net',
        user='oncologyadmin',
        password=os.getenv('AZURE_SQL_PASSWORD'),
        database='oncology-patients',
        port=1433
    )
    cursor = conn.cursor()
    cursor.execute("SELECT @@VERSION")
    version = cursor.fetchone()[0]
    print("✅ Connected to Azure SQL!")
    print(f"SQL Server: {version[:80]}")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to Azure SQL!
SQL Server: Microsoft SQL Azure (RTM) - 12.0.2000.8 
	May 28 2026 22:03:36 
	Copyright (C) 2


In [13]:
import pymssql
import os
from dotenv import load_dotenv

load_dotenv('../.env')

conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()

# Create all 5 tables
tables = {
    "Patients": """
        CREATE TABLE Patients (
            patient_id NVARCHAR(100) PRIMARY KEY,
            first_name NVARCHAR(100),
            last_name NVARCHAR(100),
            gender NVARCHAR(20),
            birth_date DATE,
            race NVARCHAR(100),
            ethnicity NVARCHAR(100),
            state NVARCHAR(50),
            dataset_source NVARCHAR(50)
        )
    """,
    "Conditions": """
        CREATE TABLE Conditions (
            condition_id NVARCHAR(100) PRIMARY KEY,
            patient_id NVARCHAR(100),
            condition_code NVARCHAR(50),
            condition_text NVARCHAR(500),
            onset_date DATE,
            abatement_date DATE,
            is_cancer BIT,
            cancer_type NVARCHAR(100)
        )
    """,
    "Treatments": """
        CREATE TABLE Treatments (
            treatment_id NVARCHAR(100) PRIMARY KEY,
            patient_id NVARCHAR(100),
            treatment_type NVARCHAR(50),
            treatment_code NVARCHAR(50),
            treatment_text NVARCHAR(500),
            start_date DATE,
            end_date DATE,
            status NVARCHAR(50)
        )
    """,
    "Observations": """
        CREATE TABLE Observations (
            observation_id NVARCHAR(100) PRIMARY KEY,
            patient_id NVARCHAR(100),
            observation_code NVARCHAR(50),
            observation_text NVARCHAR(500),
            observation_value NVARCHAR(200),
            observation_unit NVARCHAR(50),
            observation_date DATE
        )
    """,
    "Encounters": """
        CREATE TABLE Encounters (
            encounter_id NVARCHAR(100) PRIMARY KEY,
            patient_id NVARCHAR(100),
            encounter_type NVARCHAR(100),
            encounter_date DATE,
            end_date DATE,
            reason_text NVARCHAR(500)
        )
    """
}

# Create each table
for table_name, sql in tables.items():
    try:
        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        cursor.execute(sql)
        conn.commit()
        print(f"✅ Created table: {table_name}")
    except Exception as e:
        print(f"❌ Error creating {table_name}: {e}")

print("\n🎉 All tables created successfully!")
conn.close()

✅ Created table: Patients
✅ Created table: Conditions
✅ Created table: Treatments
✅ Created table: Observations
✅ Created table: Encounters

🎉 All tables created successfully!


In [14]:
import json
import os
import pymssql
from dotenv import load_dotenv
from tqdm import tqdm
from datetime import datetime

load_dotenv('../.env')

# Connect to Azure SQL
conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()
print("✅ Connected to Azure SQL!")

# Helper function to parse dates safely
def parse_date(date_str):
    if not date_str:
        return None
    try:
        return date_str[:10]  # Take just YYYY-MM-DD
    except:
        return None

# Helper to detect cancer type
cancer_keywords = {
    'breast': ['breast', 'mammary'],
    'lung': ['lung', 'bronch', 'pulmonary'],
    'colorectal': ['colon', 'rectal', 'colorectal', 'intestin']
}

def get_cancer_type(text):
    if not text:
        return None
    text_lower = text.lower()
    for cancer_type, keywords in cancer_keywords.items():
        if any(kw in text_lower for kw in keywords):
            return cancer_type
    return None

def is_cancer(text):
    cancer_terms = ['malignant', 'carcinoma', 'neoplasm',
                    'cancer', 'tumor', 'lymphoma', 'leukemia']
    if not text:
        return False
    return any(term in text.lower() for term in cancer_terms)

print("✅ Helper functions ready!")
print("Ready to start loading patients...")

✅ Connected to Azure SQL!
✅ Helper functions ready!
Ready to start loading patients...


In [15]:
def process_patient_file(filepath, dataset_source):
    """Extract all relevant data from one FHIR patient file"""
    
    with open(filepath, 'r') as f:
        bundle = json.load(f)
    
    patient_data = {
        'patient': None,
        'conditions': [],
        'treatments': [],
        'observations': [],
        'encounters': []
    }
    
    for entry in bundle.get('entry', []):
        resource = entry.get('resource', {})
        resource_type = resource.get('resourceType')
        
        # PATIENT
        if resource_type == 'Patient':
            name = resource.get('name', [{}])[0]
            given = ' '.join(name.get('given', []))
            family = name.get('family', '')
            
            # Get address state
            addresses = resource.get('address', [{}])
            state = addresses[0].get('state', '') if addresses else ''
            
            # Get race and ethnicity from extensions
            race = ''
            ethnicity = ''
            for ext in resource.get('extension', []):
                url = ext.get('url', '')
                if 'race' in url:
                    for sub in ext.get('extension', []):
                        if sub.get('url') == 'text':
                            race = sub.get('valueString', '')
                if 'ethnicity' in url:
                    for sub in ext.get('extension', []):
                        if sub.get('url') == 'text':
                            ethnicity = sub.get('valueString', '')
            
            patient_data['patient'] = {
                'patient_id': resource.get('id', ''),
                'first_name': given,
                'last_name': family,
                'gender': resource.get('gender', ''),
                'birth_date': parse_date(resource.get('birthDate')),
                'race': race,
                'ethnicity': ethnicity,
                'state': state,
                'dataset_source': dataset_source
            }
        
        # CONDITIONS
        elif resource_type == 'Condition':
            code = resource.get('code', {})
            text = code.get('text', '')
            condition_id = resource.get('id', '')
            
            patient_data['conditions'].append({
                'condition_id': condition_id,
                'condition_code': code.get('coding', [{}])[0].get('code', ''),
                'condition_text': text,
                'onset_date': parse_date(resource.get('onsetDateTime')),
                'abatement_date': parse_date(resource.get('abatementDateTime')),
                'is_cancer': 1 if is_cancer(text) else 0,
                'cancer_type': get_cancer_type(text)
            })
        
        # TREATMENTS — MedicationRequest
        elif resource_type == 'MedicationRequest':
            med = resource.get('medicationCodeableConcept', {})
            text = med.get('text', '')
            patient_data['treatments'].append({
                'treatment_id': resource.get('id', ''),
                'treatment_type': 'Medication',
                'treatment_code': med.get('coding', [{}])[0].get('code', ''),
                'treatment_text': text,
                'start_date': parse_date(resource.get('a

SyntaxError: unterminated string literal (detected at line 80) (2169021843.py, line 80)

In [16]:
def process_patient_file(filepath, dataset_source):
    """Extract all relevant data from one FHIR patient file"""
    
    with open(filepath, 'r') as f:
        bundle = json.load(f)
    
    patient_data = {
        'patient': None,
        'conditions': [],
        'treatments': [],
        'observations': [],
        'encounters': []
    }
    
    for entry in bundle.get('entry', []):
        resource = entry.get('resource', {})
        resource_type = resource.get('resourceType')
        
        # PATIENT
        if resource_type == 'Patient':
            name = resource.get('name', [{}])[0]
            given = ' '.join(name.get('given', []))
            family = name.get('family', '')
            
            # Get address state
            addresses = resource.get('address', [{}])
            state = addresses[0].get('state', '') if addresses else ''
            
            # Get race and ethnicity from extensions
            race = ''
            ethnicity = ''
            for ext in resource.get('extension', []):
                url = ext.get('url', '')
                if 'race' in url:
                    for sub in ext.get('extension', []):
                        if sub.get('url') == 'text':
                            race = sub.get('valueString', '')
                if 'ethnicity' in url:
                    for sub in ext.get('extension', []):
                        if sub.get('url') == 'text':
                            ethnicity = sub.get('valueString', '')
            
            patient_data['patient'] = {
                'patient_id': resource.get('id', ''),
                'first_name': given,
                'last_name': family,
                'gender': resource.get('gender', ''),
                'birth_date': parse_date(resource.get('birthDate')),
                'race': race,
                'ethnicity': ethnicity,
                'state': state,
                'dataset_source': dataset_source
            }
        
        # CONDITIONS
        elif resource_type == 'Condition':
            code = resource.get('code', {})
            text = code.get('text', '')
            condition_id = resource.get('id', '')
            
            patient_data['conditions'].append({
                'condition_id': condition_id,
                'condition_code': code.get('coding', [{}])[0].get('code', ''),
                'condition_text': text,
                'onset_date': parse_date(resource.get('onsetDateTime')),
                'abatement_date': parse_date(resource.get('abatementDateTime')),
                'is_cancer': 1 if is_cancer(text) else 0,
                'cancer_type': get_cancer_type(text)
            })
        
        # TREATMENTS — MedicationRequest
        elif resource_type == 'MedicationRequest':
            med = resource.get('medicationCodeableConcept', {})
            text = med.get('text', '')
            patient_data['treatments'].append({
                'treatment_id': resource.get('id', ''),
                'treatment_type': 'Medication',
                'treatment_code': med.get('coding', [{}])[0].get('code', ''),
                'treatment_text': text,
                'start_date': parse_date(resource.get('authoredOn')),
                'end_date': None,
                'status': resource.get('status', '')
            })
        
        # TREATMENTS — Procedure
        elif resource_type == 'Procedure':
            code = resource.get('code', {})
            text = code.get('text', '')
            patient_data['treatments'].append({
                'treatment_id': resource.get('id', ''),
                'treatment_type': 'Procedure',
                'treatment_code': code.get('coding', [{}])[0].get('code', ''),
                'treatment_text': text,
                'start_date': parse_date(resource.get('performedDateTime') or
                              resource.get('performedPeriod', {}).get('start')),
                'end_date': parse_date(resource.get('performedPeriod', {}).get('end')),
                'status': resource.get('status', '')
            })
        
        # OBSERVATIONS
        elif resource_type == 'Observation':
            code = resource.get('code', {})
            value = resource.get('valueQuantity', {})
            patient_data['observations'].append({
                'observation_id': resource.get('id', ''),
                'observation_code': code.get('coding', [{}])[0].get('code', ''),
                'observation_text': code.get('text', ''),
                'observation_value': str(value.get('value', '')),
                'observation_unit': value.get('unit', ''),
                'observation_date': parse_date(resource.get('effectiveDateTime'))
            })
        
        # ENCOUNTERS
        elif resource_type == 'Encounter':
            reason = resource.get('reasonCode', [{}])
            reason_text = reason[0].get('coding', [{}])[0].get('display', '') if reason else ''
            encounter_type = resource.get('type', [{}])
            type_text = encounter_type[0].get('text', '') if encounter_type else ''
            period = resource.get('period', {})
            patient_data['encounters'].append({
                'encounter_id': resource.get('id', ''),
                'encounter_type': type_text,
                'encounter_date': parse_date(period.get('start')),
                'end_date': parse_date(period.get('end')),
                'reason_text': reason_text
            })
    
    return patient_data

print("✅ ETL function ready!")
print("Now ready to load patients into Azure SQL...")

✅ ETL function ready!
Now ready to load patients into Azure SQL...


In [17]:
def insert_patient_data(cursor, conn, patient_data, patient_id):
    """Insert one patient's data into all tables"""
    
    p = patient_data['patient']
    
    # Insert Patient
    cursor.execute("""
        INSERT INTO Patients 
        (patient_id, first_name, last_name, gender, birth_date, 
         race, ethnicity, state, dataset_source)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (p['patient_id'], p['first_name'], p['last_name'],
          p['gender'], p['birth_date'], p['race'],
          p['ethnicity'], p['state'], p['dataset_source']))
    
    # Insert Conditions
    for c in patient_data['conditions']:
        cursor.execute("""
            INSERT INTO Conditions
            (condition_id, patient_id, condition_code, condition_text,
             onset_date, abatement_date, is_cancer, cancer_type)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (c['condition_id'], patient_id, c['condition_code'],
              c['condition_text'], c['onset_date'], c['abatement_date'],
              c['is_cancer'], c['cancer_type']))
    
    # Insert Treatments
    for t in patient_data['treatments']:
        cursor.execute("""
            INSERT INTO Treatments
            (treatment_id, patient_id, treatment_type, treatment_code,
             treatment_text, start_date, end_date, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (t['treatment_id'], patient_id, t['treatment_type'],
              t['treatment_code'], t['treatment_text'],
              t['start_date'], t['end_date'], t['status']))
    
    # Insert Observations (limit to 20 per patient to keep DB lean)
    for o in patient_data['observations'][:20]:
        cursor.execute("""
            INSERT INTO Observations
            (observation_id, patient_id, observation_code,
             observation_text, observation_value,
             observation_unit, observation_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (o['observation_id'], patient_id, o['observation_code'],
              o['observation_text'], o['observation_value'],
              o['observation_unit'], o['observation_date']))
    
    # Insert Encounters
    for e in patient_data['encounters']:
        cursor.execute("""
            INSERT INTO Encounters
            (encounter_id, patient_id, encounter_type,
             encounter_date, end_date, reason_text)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (e['encounter_id'], patient_id, e['encounter_type'],
              e['encounter_date'], e['end_date'], e['reason_text']))
    
    conn.commit()

# ── LOAD FIRST 100 PATIENTS ──────────────────────────────
print("Starting ETL — loading first 100 patients...")
print("=" * 50)

success = 0
errors = 0
limit = 50  # 50 from each dataset = 100 total

datasets = [
    ('../data/raw/breast-cancer', 'breast-cancer'),
    ('../data/raw/mixed-cancer', 'mixed-cancer')
]

for folder, source in datasets:
    files = get_patient_files(folder)[:limit]
    print(f"\nLoading {len(files)} patients from {source}...")
    
    for filename in tqdm(files, desc=source):
        filepath = os.path.join(folder, filename)
        try:
            data = process_patient_file(filepath, source)
            if data['patient']:
                patient_id = data['patient']['patient_id']
                insert_patient_data(cursor, conn, data, patient_id)
                success += 1
        except Exception as e:
            errors += 1

print(f"\n{'='*50}")
print(f"✅ Successfully loaded: {success} patients")
print(f"❌ Errors: {errors} patients")
print(f"Total processed: {success + errors}")

Starting ETL — loading first 100 patients...

Loading 50 patients from breast-cancer...


breast-cancer: 100%|██████████| 50/50 [23:49<00:00, 28.60s/it] 



Loading 50 patients from mixed-cancer...


mixed-cancer: 100%|██████████| 50/50 [00:00<00:00, 100.92it/s]


✅ Successfully loaded: 46 patients
❌ Errors: 54 patients
Total processed: 100


In [18]:
# Check what error is happening
import os
import json

# Test one file to see the error
folder = '../data/raw/breast-cancer'
files = get_patient_files(folder)

for filename in files[:10]:
    filepath = os.path.join(folder, filename)
    try:
        data = process_patient_file(filepath, 'breast-cancer')
        if data['patient']:
            patient_id = data['patient']['patient_id']
            insert_patient_data(cursor, conn, data, patient_id)
            print(f"✅ {filename[:40]}")
    except Exception as e:
        print(f"❌ {filename[:40]}")
        print(f"   Error: {e}")
        break  # Stop at first error to see it clearly

❌ Katherine209_Reichert620_8c0d6e0a-2cbe-4
   Error: Not connected to any MS SQL server


In [19]:
# Reconnect to Azure SQL
conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()
print("✅ Reconnected to Azure SQL!")

# Check how many patients already loaded
cursor.execute("SELECT COUNT(*) FROM Patients")
count = cursor.fetchone()[0]
print(f"Patients already in database: {count}")

# Check which patient IDs already exist
cursor.execute("SELECT patient_id FROM Patients")
existing_ids = set(row[0] for row in cursor.fetchall())
print(f"Existing patient IDs: {len(existing_ids)}")

✅ Reconnected to Azure SQL!
Patients already in database: 46
Existing patient IDs: 46


In [21]:
# Load remaining patients — skip already loaded ones
print("Loading remaining patients (skipping existing)...")
print("=" * 50)

success = 0
errors = 0
skipped = 0

datasets = [
    ('../data/raw/breast-cancer', 'breast-cancer'),
    ('../data/raw/mixed-cancer', 'mixed-cancer')
]

for folder, source in datasets:
    files = get_patient_files(folder)[:50]
    print(f"\nProcessing {source}...")
    
    for filename in tqdm(files, desc=source):
        filepath = os.path.join(folder, filename)
        try:
            data = process_patient_file(filepath, source)
            if data['patient']:
                patient_id = data['patient']['patient_id']
                
                # Skip if already loaded
                if patient_id in existing_ids:
                    skipped += 1
                    continue
                
                # Reconnect if needed
                try:
                    insert_patient_data(cursor, conn, data, patient_id)
                    existing_ids.add(patient_id)
                    success += 1
                except Exception as e:
                    if 'Not connected' in str(e):
                        # Reconnect and retry
                        conn = pymssql.connect(
                            server='oncology-sql-server.database.windows.net',
                            user='oncologyadmin',
                            password=os.getenv('AZURE_SQL_PASSWORD'),
                            database='oncology-patients',
                            port=1433
                        )
                        cursor = conn.cursor()
                        insert_patient_data(cursor, conn, data, patient_id)
                        existing_ids.add(patient_id)
                        success += 1
                    else:
                        errors += 1
        except Exception as e:
            errors += 1

# Final count
cursor.execute("SELECT COUNT(*) FROM Patients")
total = cursor.fetchone()[0]

print(f"\n{'='*50}")
print(f"✅ Newly loaded: {success} patients")
print(f"⏭️  Skipped (already in DB): {skipped} patients")
print(f"❌ Errors: {errors} patients")
print(f"📊 Total patients in database: {total}")

Loading remaining patients (skipping existing)...

Processing breast-cancer...


breast-cancer: 100%|██████████| 50/50 [54:11<00:00, 65.04s/it] 



Processing mixed-cancer...


mixed-cancer: 100%|██████████| 50/50 [1:19:35<00:00, 95.51s/it]    


✅ Newly loaded: 4 patients
⏭️  Skipped (already in DB): 91 patients
❌ Errors: 5 patients
📊 Total patients in database: 95


In [22]:
# Validate data with cohort queries
print("=" * 50)
print("COHORT VALIDATION QUERIES")
print("=" * 50)

# 1. Total patients
cursor.execute("SELECT COUNT(*) FROM Patients")
print(f"\n📊 Total patients: {cursor.fetchone()[0]}")

# 2. Patients by gender
cursor.execute("""
    SELECT gender, COUNT(*) as count 
    FROM Patients 
    GROUP BY gender
""")
print("\n👥 Patients by gender:")
for row in cursor.fetchall():
    print(f"   {row[0]}: {row[1]}")

# 3. Cancer patients by type
cursor.execute("""
    SELECT cancer_type, COUNT(DISTINCT patient_id) as patients
    FROM Conditions
    WHERE is_cancer = 1 AND cancer_type IS NOT NULL
    GROUP BY cancer_type
    ORDER BY patients DESC
""")
print("\n🎗️ Cancer patients by type:")
for row in cursor.fetchall():
    print(f"   {row[0]}: {row[1]} patients")

# 4. Total conditions loaded
cursor.execute("SELECT COUNT(*) FROM Conditions")
print(f"\n🏥 Total conditions: {cursor.fetchone()[0]}")

# 5. Total treatments loaded
cursor.execute("SELECT COUNT(*) FROM Treatments")
print(f"💊 Total treatments: {cursor.fetchone()[0]}")

# 6. Total observations loaded
cursor.execute("SELECT COUNT(*) FROM Observations")
print(f"🔬 Total observations: {cursor.fetchone()[0]}")

# 7. Total encounters loaded
cursor.execute("SELECT COUNT(*) FROM Encounters")
print(f"🏨 Total encounters: {cursor.fetchone()[0]}")

print("\n" + "=" * 50)
print("✅ Validation complete!")

COHORT VALIDATION QUERIES

📊 Total patients: 95

👥 Patients by gender:
   female: 67
   male: 28

🎗️ Cancer patients by type:
   breast: 48 patients
   colorectal: 28 patients
   lung: 13 patients

🏥 Total conditions: 4712
💊 Total treatments: 25940
🔬 Total observations: 1900
🏨 Total encounters: 9148

✅ Validation complete!


In [1]:
import requests
from bs4 import BeautifulSoup
import json
import os

# Create folder for guidelines
os.makedirs('../data/raw/guidelines', exist_ok=True)

# NCI PDQ URLs for our 3 cancer types
guidelines = {
    'breast_cancer': {
        'url': 'https://www.cancer.gov/types/breast/patient/breast-treatment-pdq',
        'cancer_type': 'breast',
        'filename': 'breast_cancer_pdq.txt'
    },
    'lung_cancer': {
        'url': 'https://www.cancer.gov/types/lung/patient/non-small-cell-lung-treatment-pdq',
        'cancer_type': 'lung',
        'filename': 'lung_cancer_pdq.txt'
    },
    'colorectal_cancer': {
        'url': 'https://www.cancer.gov/types/colorectal/patient/colon-treatment-pdq',
        'cancer_type': 'colorectal',
        'filename': 'colorectal_cancer_pdq.txt'
    }
}

print("Downloading NCI PDQ guidelines...")
print("=" * 50)

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'
}

for name, info in guidelines.items():
    try:
        response = requests.get(info['url'], headers=headers, timeout=30)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract main content
        content = soup.find('div', {'class': 'contentzone'})
        if not content:
            content = soup.find('main')
        if not content:
            content = soup.find('body')
            
        # Get clean text
        text = content.get_text(separator='\n', strip=True)
        
        # Save to file
        filepath = f"../data/raw/guidelines/{info['filename']}"
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"Cancer Type: {info['cancer_type']}\n")
            f.write(f"Source: {info['url']}\n")
            f.write(f"=" * 50 + "\n\n")
            f.write(text)
        
        print(f"✅ {name}: {len(text):,} characters saved")
        
    except Exception as e:
        print(f"❌ {name}: Error — {e}")

print("\n✅ Download complete!")

✅ breast_cancer: 3,530 characters saved
✅ lung_cancer: 57,288 characters saved
✅ colorectal_cancer: 38,250 characters saved

✅ Download complete!


In [2]:
pip install beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [beautifulsoup4]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Check what we downloaded
import os

guidelines_folder = '../data/raw/guidelines'
files = os.listdir(guidelines_folder)

print("Downloaded guideline files:")
print("=" * 50)

for filename in files:
    filepath = os.path.join(guidelines_folder, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Count words
    words = len(content.split())
    chars = len(content)
    lines = len(content.split('\n'))
    
    print(f"\n📄 {filename}")
    print(f"   Characters: {chars:,}")
    print(f"   Words: {words:,}")
    print(f"   Lines: {lines:,}")
    print(f"   First 200 chars:")
    print(f"   {content[100:300]}")

Downloaded guideline files:

📄 colorectal_cancer_pdq.txt
   Characters: 38,402
   Words: 6,451
   Lines: 809
   First 200 chars:

Colon Cancer Treatment (PDQ®)–Patient Version
On This Page
General Information About Colon Cancer
Stages of Colon Cancer
Treatment Option Overview
T

📄 lung_cancer_pdq.txt
   Characters: 57,442
   Words: 9,761
   Lines: 1,155
   First 200 chars:
   q

Non-Small Cell Lung Cancer Treatment  (PDQ®)–Patient Version
On This Page
General Information About Non-Small Cell Lung Cancer
Stages of Non-Small

📄 breast_cancer_pdq.txt
   Characters: 3,675
   Words: 552
   Lines: 53
   First 200 chars:

Breast Cancer Treatment
Different types of treatment are available for breast cancer. You and your cancer care team will work together to decide your treat


In [3]:
# Get richer breast cancer content 
# from NCI health professional version
import requests
from bs4 import BeautifulSoup

breast_urls = [
    'https://www.cancer.gov/types/breast/hp/breast-treatment-pdq',
    'https://www.cancer.gov/types/breast/patient/breast-treatment-pdq#section/_1',
]

all_text = ""
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}

for url in breast_urls:
    try:
        response = requests.get(url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove navigation and script elements
        for tag in soup(['script', 'style', 'nav', 'header', 'footer']):
            tag.decompose()
        
        content = soup.find('main') or soup.find('body')
        text = content.get_text(separator='\n', strip=True)
        all_text += text + "\n\n"
        print(f"✅ Fetched: {url[:60]}")
        print(f"   Characters: {len(text):,}")
    except Exception as e:
        print(f"❌ Error: {e}")

# Save enriched breast cancer file
filepath = '../data/raw/guidelines/breast_cancer_pdq.txt'
with open(filepath, 'w', encoding='utf-8') as f:
    f.write(f"Cancer Type: breast\n")
    f.write(f"Source: NCI PDQ Breast Cancer Treatment\n")
    f.write("=" * 50 + "\n\n")
    f.write(all_text)

print(f"\n✅ Breast cancer guidelines updated!")
print(f"   Total characters: {len(all_text):,}")
print(f"   Total words: {len(all_text.split()):,}")

✅ Fetched: https://www.cancer.gov/types/breast/hp/breast-treatment-pdq
   Characters: 428,710
✅ Fetched: https://www.cancer.gov/types/breast/patient/breast-treatment
   Characters: 3,530

✅ Breast cancer guidelines updated!
   Total characters: 432,244
   Total words: 65,380


In [4]:
import tiktoken
import json
import os

# Load tokenizer for text-embedding-3-small
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_document(text, cancer_type, source_url, 
                   chunk_size=600, overlap=100):
    """
    Split document into overlapping chunks with metadata.
    Target: 500-800 tokens per chunk with 100 token overlap.
    """
    # Clean the text
    lines = [line.strip() for line in text.split('\n') 
             if line.strip()]
    clean_text = ' '.join(lines)
    
    # Tokenize
    tokens = tokenizer.encode(clean_text)
    total_tokens = len(tokens)
    
    chunks = []
    start = 0
    chunk_num = 0
    
    while start < total_tokens:
        # Get chunk tokens
        end = min(start + chunk_size, total_tokens)
        chunk_tokens = tokens[start:end]
        
        # Decode back to text
        chunk_text = tokenizer.decode(chunk_tokens)
        
        # Create chunk with metadata
        chunk = {
            'chunk_id': f"{cancer_type}_{chunk_num:04d}",
            'text': chunk_text,
            'cancer_type': cancer_type,
            'source_url': source_url,
            'token_count': len(chunk_tokens),
            'chunk_number': chunk_num
        }
        chunks.append(chunk)
        
        # Move forward with overlap
        start += chunk_size - overlap
        chunk_num += 1
    
    return chunks

# Test on colorectal cancer first (smallest file)
guidelines_folder = '../data/raw/guidelines'

source_urls = {
    'breast': 'https://www.cancer.gov/types/breast/hp/breast-treatment-pdq',
    'lung': 'https://www.cancer.gov/types/lung/patient/non-small-cell-lung-treatment-pdq',
    'colorectal': 'https://www.cancer.gov/types/colorectal/patient/colon-treatment-pdq'
}

files = {
    'breast': 'breast_cancer_pdq.txt',
    'lung': 'lung_cancer_pdq.txt',
    'colorectal': 'colorectal_cancer_pdq.txt'
}

all_chunks = []

print("Chunking guidelines...")
print("=" * 50)

for cancer_type, filename in files.items():
    filepath = os.path.join(guidelines_folder, filename)
    
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    chunks = chunk_document(
        text=text,
        cancer_type=cancer_type,
        source_url=source_urls[cancer_type]
    )
    
    all_chunks.extend(chunks)
    print(f"✅ {cancer_type}: {len(chunks)} chunks")

print(f"\n📊 Total chunks: {len(all_chunks)}")
print(f"Average tokens per chunk: {sum(c['token_count'] for c in all_chunks) // len(all_chunks)}")

# Save chunks to file
chunks_path = '../data/processed/guideline_chunks.json'
os.makedirs('../data/processed', exist_ok=True)
with open(chunks_path, 'w') as f:
    json.dump(all_chunks, f, indent=2)

print(f"\n✅ Chunks saved to {chunks_path}")

Chunking guidelines...
✅ breast: 230 chunks
✅ lung: 25 chunks
✅ colorectal: 17 chunks

📊 Total chunks: 272
Average tokens per chunk: 593

✅ Chunks saved to ../data/processed/guideline_chunks.json


In [5]:
from openai import AzureOpenAI
import json
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv('../.env')

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv('AZURE_SEARCH_API_KEY'),
    api_version="2024-02-01",
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)

# Load chunks
with open('../data/processed/guideline_chunks.json', 'r') as f:
    chunks = json.load(f)

print(f"Generating embeddings for {len(chunks)} chunks...")
print("This will take 2-3 minutes...")
print("=" * 50)

embedded_chunks = []
errors = 0

for i, chunk in enumerate(tqdm(chunks, desc="Embedding")):
    try:
        response = client.embeddings.create(
            input=chunk['text'],
            model=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
        )
        
        # Add embedding to chunk
        chunk['embedding'] = response.data[0].embedding
        embedded_chunks.append(chunk)
        
        # Small delay to avoid rate limiting
        if i % 10 == 0 and i > 0:
            time.sleep(0.5)
            
    except Exception as e:
        errors += 1
        print(f"❌ Error on chunk {i}: {e}")

print(f"\n✅ Successfully embedded: {len(embedded_chunks)} chunks")
print(f"❌ Errors: {errors}")
print(f"Embedding dimensions: {len(embedded_chunks[0]['embedding'])}")

# Save embedded chunks
embedded_path = '../data/processed/embedded_chunks.json'
with open(embedded_path, 'w') as f:
    json.dump(embedded_chunks, f)

print(f"✅ Saved to {embedded_path}")

Generating embeddings for 272 chunks...
This will take 2-3 minutes...


Embedding:   3%|▎         | 7/272 [00:00<00:15, 17.07it/s]

❌ Error on chunk 0: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 1: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 2: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 3: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscri

Embedding:   7%|▋         | 19/272 [00:00<00:07, 35.55it/s]

❌ Error on chunk 11: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 12: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 13: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 14: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active sub

...

Embedding:  97%|█████████▋| 263/272 [00:05<00:00, 53.41it/s]

❌ Error on chunk 253: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 254: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 255: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 256: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active

Embedding: 100%|██████████| 272/272 [00:05<00:00, 46.86it/s]


❌ Error on chunk 265: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 266: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 267: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}
❌ Error on chunk 268: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active

IndexError: list index out of range

In [8]:
from openai import AzureOpenAI
import json
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv('../.env')

# Check what keys we have
print("Checking environment variables...")
print(f"OpenAI endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Embedding deployment: {os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')}")

# Get the API key from Foundry portal
# We need the Azure OpenAI API key — not the Search key
# Let's use the project endpoint instead

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

try:
    # Try using project client
    client = AIProjectClient(
        endpoint=os.getenv('AZURE_AI_PROJECT_ENDPOINT'),
        credential=DefaultAzureCredential()
    )
    openai_client = client.inference.get_azure_openai_client(api_version="2024-02-01")
    print("✅ Connected via AIProjectClient!")
    
except Exception as e:
    print(f"❌ AIProjectClient failed: {e}")
    print("Trying direct API key...")
    
    # Fall back to direct key from .env
    # Go to ai.azure.com → your project → Settings → API keys
    # Add this to your .env: AZURE_OPENAI_API_KEY=your_key_here
    api_key = os.getenv('AZURE_OPENAI_API_KEY', '')
    if not api_key:
        print("⚠️  AZURE_OPENAI_API_KEY not in .env")
        print("Go to: ai.azure.com → oncology-care-insights → Settings")
        print("Copy the API key and add to .env:")
        print("AZURE_OPENAI_API_KEY=your_key_here")
    else:
        openai_client = AzureOpenAI(
            api_key=api_key,
            api_version="2024-02-01",
            azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
        )
        print("✅ Connected via direct API key!")

Checking environment variables...
OpenAI endpoint: https://oncology-care-insights-resource.openai.azure.com
Embedding deployment: text-embedding-3-small
❌ AIProjectClient failed: 'AIProjectClient' object has no attribute 'inference'
Trying direct API key...
✅ Connected via direct API key!


In [9]:
import time
from tqdm import tqdm

# Load chunks
with open('../data/processed/guideline_chunks.json', 'r') as f:
    chunks = json.load(f)

print(f"Generating embeddings for {len(chunks)} chunks...")
print("=" * 50)

embedded_chunks = []
errors = 0

for i, chunk in enumerate(tqdm(chunks, desc="Embedding")):
    try:
        response = openai_client.embeddings.create(
            input=chunk['text'],
            model=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
        )
        
        chunk['embedding'] = response.data[0].embedding
        embedded_chunks.append(chunk)
        
        # Small delay every 10 chunks
        if i % 10 == 0 and i > 0:
            time.sleep(0.5)
            
    except Exception as e:
        errors += 1
        print(f"❌ Chunk {i}: {e}")

print(f"\n✅ Successfully embedded: {len(embedded_chunks)} chunks")
print(f"❌ Errors: {errors}")

if embedded_chunks:
    print(f"Embedding dimensions: {len(embedded_chunks[0]['embedding'])}")
    
    # Save embedded chunks
    embedded_path = '../data/processed/embedded_chunks.json'
    with

SyntaxError: invalid syntax (2520171699.py, line 40)

In [10]:
import time
from tqdm import tqdm

# Load chunks
with open('../data/processed/guideline_chunks.json', 'r') as f:
    chunks = json.load(f)

print(f"Generating embeddings for {len(chunks)} chunks...")
print("=" * 50)

embedded_chunks = []
errors = 0

for i, chunk in enumerate(tqdm(chunks, desc="Embedding")):
    try:
        response = openai_client.embeddings.create(
            input=chunk['text'],
            model=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
        )
        
        chunk['embedding'] = response.data[0].embedding
        embedded_chunks.append(chunk)
        
        # Small delay every 10 chunks
        if i % 10 == 0 and i > 0:
            time.sleep(0.5)
            
    except Exception as e:
        errors += 1
        print(f"❌ Chunk {i}: {e}")

print(f"\n✅ Successfully embedded: {len(embedded_chunks)} chunks")
print(f"❌ Errors: {errors}")

if embedded_chunks:
    print(f"Embedding dimensions: {len(embedded_chunks[0]['embedding'])}")
    
    # Save embedded chunks
    embedded_path = '../data/processed/embedded_chunks.json'
    with open(embedded_path, 'w') as f:
        json.dump(embedded_chunks, f)
    print(f"✅ Saved to {embedded_path}")

Generating embeddings for 272 chunks...


Embedding: 100%|██████████| 272/272 [01:17<00:00,  3.49it/s]



✅ Successfully embedded: 272 chunks
❌ Errors: 0
Embedding dimensions: 1536
✅ Saved to ../data/processed/embedded_chunks.json


In [11]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchFieldDataType,
    SearchableField, SearchField, VectorSearch,
    HnswAlgorithmConfiguration, VectorSearchProfile
)
from azure.core.credentials import AzureKeyCredential
import json
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Connect to Azure AI Search
search_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
search_key = os.getenv('AZURE_SEARCH_API_KEY')
index_name = os.getenv('AZURE_SEARCH_INDEX_NAME')

credential = AzureKeyCredential(search_key)
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=credential
)

print(f"✅ Connected to Azure AI Search!")
print(f"Endpoint: {search_endpoint}")
print(f"Index name: {index_name}")

# Create the index schema
fields = [
    SimpleField(name="chunk_id", type=SearchFieldDataType.String, 
                key=True),
    SearchableField(name="text", type=SearchFieldDataType.String),
    SimpleField(name="cancer_type", type=SearchFieldDataType.String,
                filterable=True),
    SimpleField(name="source_url", type=SearchFieldDataType.String),
    SimpleField(name="chunk_number", type=SearchFieldDataType.Int32),
    SimpleField(name="token_count", type=SearchFieldDataType.Int32),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="my-vector-profile"
    )
]

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="my-hnsw")],
    profiles=[VectorSearchProfile(
        name="my-vector-profile",
        algorithm_configuration_name="my-hnsw"
    )]
)

# Create index
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

try:
    index_client.create_or_update_index(index)
    print(f"✅ Index '{index_name}' created successfully!")
except Exception as e:
    print(f"❌ Index creation failed: {e}")

✅ Connected to Azure AI Search!
Endpoint: https://oncology-search.search.windows.net
Index name: oncology-guidelines
✅ Index 'oncology-guidelines' created successfully!


In [12]:
from azure.search.documents import SearchClient
from tqdm import tqdm

# Load embedded chunks
with open('../data/processed/embedded_chunks.json', 'r') as f:
    embedded_chunks = json.load(f)

print(f"Uploading {len(embedded_chunks)} chunks to Azure AI Search...")
print("=" * 50)

# Create search client
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=credential
)

# Upload in batches of 50
batch_size = 50
success = 0
errors = 0

for i in tqdm(range(0, len(embedded_chunks), batch_size), 
              desc="Uploading"):
    batch = embedded_chunks[i:i + batch_size]
    
    # Format for Azure AI Search
    documents = []
    for chunk in batch:
        documents.append({
            'chunk_id': chunk['chunk_id'],
            'text': chunk['text'],
            'cancer_type': chunk['cancer_type'],
            'source_url': chunk['source_url'],
            'chunk_number': chunk['chunk_number'],
            'token_count': chunk['token_count'],
            'embedding': chunk['embedding']
        })
    
    try:
        result = search_client.upload_documents(documents)
        success += len(documents)
    except Exception as e:
        errors += len(documents)
        print(f"❌ Batch error: {e}")

print(f"\n✅ Successfully uploaded: {success} chunks")
print(f"❌ Errors: {errors}")
print(f"\n🎉 Azure AI Search index is ready!")
print(f"Index: {index_name}")
print(f"Total chunks: {success}")

Uploading 272 chunks to Azure AI Search...


Uploading: 100%|██████████| 6/6 [00:05<00:00,  1.19it/s]


✅ Successfully uploaded: 272 chunks
❌ Errors: 0

🎉 Azure AI Search index is ready!
Index: oncology-guidelines
Total chunks: 272


In [13]:
from azure.search.documents.models import VectorizedQuery

# Test retrieval!
test_questions = [
    "What is the treatment for Stage II breast cancer?",
    "What are the symptoms of lung cancer?",
    "How is colorectal cancer diagnosed?"
]

print("Testing retrieval...")
print("=" * 50)

for question in test_questions:
    # Embed the question
    response = openai_client.embeddings.create(
        input=question,
        model=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    )
    query_vector = response.data[0].embedding
    
    # Search
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=2,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["chunk_id", "cancer_type", "text"],
        top=2
    )
    
    print(f"\n❓ {question}")
    for result in results:
        print(f"  ✅ [{result['cancer_type']}] {result['text'][:150]}...")
    print()

Testing retrieval...

❓ What is the treatment for Stage II breast cancer?
  ✅ [breast]  mastectomy: radiotherapeutic management. Int J Radiat Oncol Biol Phys 19 (4): 851-8, 1990. [PUBMED Abstract] Aebi S, Gelber S, Anderson SJ, et al.: C...
  ✅ [breast] alateral breast cancer in postmenopausal women with locally excised ductal carcinoma in situ (IBIS-II DCIS): a double-blind, randomised controlled tri...


❓ What are the symptoms of lung cancer?
  ✅ [lung]  genetics, age, and family history . Learning about risk factors for lung cancer can help you make changes that might lower your risk of getting it. S...
  ✅ [lung] Cancer Type: lung Source: https://www.cancer.gov/types/lung/patient/non-small-cell-lung-treatment-pdq ================================================...


❓ How is colorectal cancer diagnosed?
  ✅ [colorectal]  bulging tissue), abnormal areas, or cancer. A colonoscope is a thin, tube-like instrument with a light and a lens for viewing. It may also have a  to...
  ✅ [colo

In [14]:
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Test basic connection to Foundry
endpoint = os.getenv('AZURE_AI_PROJECT_ENDPOINT')
api_key = os.getenv('AZURE_OPENAI_API_KEY')

print(f"Project endpoint: {endpoint}")
print(f"API key present: {'Yes' if api_key else 'No'}")

# Test with OpenAI client directly
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=api_key,
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)

# Simple test call
response = client.chat.completions.create(
    model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
    messages=[
        {"role": "system", "content": "You are an oncology assistant."},
        {"role": "user", "content": "What is breast cancer in one sentence?"}
    ],
    max_tokens=100
)

print(f"\n✅ Model responding!")
print(f"Response: {response.choices[0].message.content}")

Project endpoint: https://oncology-care-insights-resource.services.ai.azure.com
API key present: Yes


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}

In [15]:
import pymssql
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Reconnect to Azure SQL
conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()
print("✅ Connected to Azure SQL!")

# Define cohort query functions
def get_cohort_stats(cancer_type=None):
    """Get patient counts by cancer type"""
    if cancer_type:
        cursor.execute("""
            SELECT 
                cancer_type,
                COUNT(DISTINCT patient_id) as patient_count,
                SUM(CASE WHEN gender='female' THEN 1 ELSE 0 END) as female_count,
                SUM(CASE WHEN gender='male' THEN 1 ELSE 0 END) as male_count
            FROM Conditions c
            JOIN Patients p ON c.patient_id = p.patient_id
            WHERE is_cancer = 1 
            AND cancer_type = %s
            GROUP BY cancer_type
        """, (cancer_type,))
    else:
        cursor.execute("""
            SELECT 
                cancer_type,
                COUNT(DISTINCT patient_id) as patient_count,
                SUM(CASE WHEN gender='female' THEN 1 ELSE 0 END) as female_count,
                SUM(CASE WHEN gender='male' THEN 1 ELSE 0 END) as male_count
            FROM Conditions c
            JOIN Patients p ON c.patient_id = p.patient_id
            WHERE is_cancer = 1
            AND cancer_type IS NOT NULL
            GROUP BY cancer_type
            ORDER BY patient_count DESC
        """)
    
    results = cursor.fetchall()
    return results

def get_treatment_distribution(cancer_type):
    """Get top treatments for a cancer type"""
    cursor.execute("""
        SELECT TOP 10
            t.treatment_text,
            t.treatment_type,
            COUNT(*) as count
        FROM Treatments t
        JOIN Conditions c ON t.patient_id = c.patient_id
        WHERE c.is_cancer = 1
        AND c.cancer_type = %s
        AND t.treatment_text IS NOT NULL
        AND t.treatment_text != ''
        GROUP BY t.treatment_text, t.treatment_type
        ORDER BY count DESC
    """, (cancer_type,))
    return cursor.fetchall()

def get_age_distribution(cancer_type):
    """Get age distribution for a cancer type"""
    cursor.execute("""
        SELECT 
            CASE 
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 40 THEN 'Under 40'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 50 THEN '40-49'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 60 THEN '50-59'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 70 THEN '60-69'
                ELSE '70+'
            END as age_group,
            COUNT(DISTINCT p.patient_id) as count
        FROM Patients p
        JOIN Conditions c ON p.patient_id = c.patient_id
        WHERE c.is_cancer = 1
        AND c.cancer_type = %s
        GROUP BY 
            CASE 
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 40 THEN 'Under 40'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 50 THEN '40-49'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 60 THEN '50-59'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 70 THEN '60-69'
                ELSE '70+'
            END
        ORDER BY count DESC
    """, (cancer_type,))
    return cursor.fetchall()

print("✅ Cohort query functions ready!")
print("\nTesting functions...")

# Test all 3 functions
print("\n📊 Cancer patients by type:")
for row in get_cohort_stats():
    print(f"  {row[0]}: {row[1]} patients ({row[2]}F / {row[3]}M)")

print("\n💊 Top breast cancer treatments:")
for row in get_treatment_distribution('breast')[:5]:
    print(f"  {row[0][:50]}: {row[2]} patients")

print("\n👥 Breast cancer age distribution:")
for row in get_age_distribution('breast'):
    print(f"  {row[0]}: {row[1]} patients")

✅ Connected to Azure SQL!
✅ Cohort query functions ready!

Testing functions...

📊 Cancer patients by type:


OperationalError: (209, b"Ambiguous column name 'patient_id'.DB-Lib error message 20018, severity 16:\nGeneral SQL Server error: Check messages from the SQL Server\n")

In [16]:
import pymssql
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Reconnect to Azure SQL
conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()
print("✅ Connected to Azure SQL!")

def get_cohort_stats(cancer_type=None):
    """Get patient counts by cancer type"""
    if cancer_type:
        cursor.execute("""
            SELECT 
                c.cancer_type,
                COUNT(DISTINCT c.patient_id) as patient_count,
                SUM(CASE WHEN p.gender='female' THEN 1 ELSE 0 END) as female_count,
                SUM(CASE WHEN p.gender='male' THEN 1 ELSE 0 END) as male_count
            FROM Conditions c
            JOIN Patients p ON c.patient_id = p.patient_id
            WHERE c.is_cancer = 1 
            AND c.cancer_type = %s
            GROUP BY c.cancer_type
        """, (cancer_type,))
    else:
        cursor.execute("""
            SELECT 
                c.cancer_type,
                COUNT(DISTINCT c.patient_id) as patient_count,
                SUM(CASE WHEN p.gender='female' THEN 1 ELSE 0 END) as female_count,
                SUM(CASE WHEN p.gender='male' THEN 1 ELSE 0 END) as male_count
            FROM Conditions c
            JOIN Patients p ON c.patient_id = p.patient_id
            WHERE c.is_cancer = 1
            AND c.cancer_type IS NOT NULL
            GROUP BY c.cancer_type
            ORDER BY patient_count DESC
        """)
    return cursor.fetchall()

def get_treatment_distribution(cancer_type):
    """Get top treatments for a cancer type"""
    cursor.execute("""
        SELECT TOP 10
            t.treatment_text,
            t.treatment_type,
            COUNT(*) as count
        FROM Treatments t
        JOIN Conditions c ON t.patient_id = c.patient_id
        WHERE c.is_cancer = 1
        AND c.cancer_type = %s
        AND t.treatment_text IS NOT NULL
        AND t.treatment_text != ''
        GROUP BY t.treatment_text, t.treatment_type
        ORDER BY count DESC
    """, (cancer_type,))
    return cursor.fetchall()

def get_age_distribution(cancer_type):
    """Get age distribution for a cancer type"""
    cursor.execute("""
        SELECT 
            CASE 
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 40 THEN 'Under 40'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 50 THEN '40-49'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 60 THEN '50-59'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 70 THEN '60-69'
                ELSE '70+'
            END as age_group,
            COUNT(DISTINCT p.patient_id) as count
        FROM Patients p
        JOIN Conditions c ON p.patient_id = c.patient_id
        WHERE c.is_cancer = 1
        AND c.cancer_type = %s
        GROUP BY 
            CASE 
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 40 THEN 'Under 40'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 50 THEN '40-49'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 60 THEN '50-59'
                WHEN DATEDIFF(year, p.birth_date, GETDATE()) < 70 THEN '60-69'
                ELSE '70+'
            END
        ORDER BY count DESC
    """, (cancer_type,))
    return cursor.fetchall()

print("✅ Functions ready! Testing...")

# Test all 3
print("\n📊 Cancer patients by type:")
for row in get_cohort_stats():
    print(f"  {row[0]}: {row[1]} patients ({row[2]}F / {row[3]}M)")

print("\n💊 Top breast cancer treatments:")
for row in get_treatment_distribution('breast')[:5]:
    print(f"  {row[0][:50]}: {row[2]} patients")

print("\n👥 Breast cancer age distribution:")
for row in get_age_distribution('breast'):
    print(f"  {row[0]}: {row[1]} patients")

✅ Connected to Azure SQL!
✅ Functions ready! Testing...

📊 Cancer patients by type:
  breast: 48 patients (42F / 6M)
  colorectal: 28 patients (18F / 10M)
  lung: 13 patients (9F / 30M)

💊 Top breast cancer treatments:
  amLODIPine 2.5 MG Oral Tablet: 782 patients
  Hospice care (regime/therapy): 666 patients
  Assessment of health and social care needs (proced: 615 patients
  insulin human, isophane 70 UNT/ML / Regular Insuli: 540 patients
  Hydrochlorothiazide 25 MG Oral Tablet: 532 patients

👥 Breast cancer age distribution:
  Under 40: 19 patients
  70+: 12 patients
  60-69: 8 patients
  50-59: 5 patients
  40-49: 4 patients


In [17]:
from openai import AzureOpenAI
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# Initialize OpenAI client
openai_client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)

def answer_oncology_question(question):
    """
    Combined agent function:
    1. Gets relevant cohort stats from Azure SQL
    2. Sends question + stats to gpt-5-mini
    3. Returns grounded answer
    """
    
    # Step 1 — Get all cohort stats
    cohort_data = get_cohort_stats()
    cohort_summary = "\n".join([
        f"- {row[0]} cancer: {row[1]} patients ({row[2]} female, {row[3]} male)"
        for row in cohort_data
    ])
    
    # Step 2 — Get specific cancer stats if mentioned
    extra_context = ""
    for cancer_type in ['breast', 'lung', 'colorectal']:
        if cancer_type in question.lower():
            age_data = get_age_distribution(cancer_type)
            age_summary = ", ".join([f"{row[0]}: {row[1]}" for row in age_data])
            extra_context = f"\n{cancer_type.title()} cancer age distribution: {age_summary}"
    
    # Step 3 — Build augmented prompt
    augmented_prompt = f"""
You are the Oncology Care Insights Agent — decision support for clinical ops analysts.

SYNTHETIC PATIENT COHORT DATA (Synthea/mCODE — not real patients):
{cohort_summary}
{extra_context}

IMPORTANT: This data is fully synthetic. Always state this when reporting numbers.

User question: {question}

Answer the question using the cohort data above where relevant.
Keep your answer concise and always cite that data is synthetic.
"""
    
    # Step 4 — Call gpt-5-mini
    response = openai_client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
        messages=[
            {"role": "user", "content": augmented_prompt}
        ],
        max_completion_tokens=500
    )
    
    return response.choices[0].message.content

# Test combined questions!
print("=" * 60)
print("COMBINED AGENT TEST")
print("=" * 60)

questions = [
    "How many breast cancer patients do we have and what is their age breakdown?",
    "Compare our patient counts across all 3 cancer types",
    "How many lung cancer patients do we have?"
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {answer_oncology_question(q)}")
    print("-" * 60)

COMBINED AGENT TEST

❓ How many breast cancer patients do we have and what is their age breakdown?
💬 You have 48 breast cancer patients in this cohort. Age breakdown (synthetic data):
- Under 40: 19
- 40–49: 4
- 50–59: 5
- 60–69: 8
- 70+: 12

This dataset is fully synthetic (Synthea/mCODE) and not real patient data.
------------------------------------------------------------

❓ Compare our patient counts across all 3 cancer types
💬 
------------------------------------------------------------

❓ How many lung cancer patients do we have?
💬 This is fully synthetic data. The cohort includes 13 lung cancer patients. (Note: the source lists gender as 9 female and 30 male—which is inconsistent with the total; the age distribution 70+: 9 and 60–69: 4 sums to 13.)
------------------------------------------------------------


In [18]:
# Fix - test comparison question directly
question = "Compare our patient counts across all 3 cancer types"

cohort_data = get_cohort_stats()
cohort_summary = "\n".join([
    f"- {row[0]} cancer: {row[1]} patients ({row[2]} female, {row[3]} male)"
    for row in cohort_data
])

response = openai_client.chat.completions.create(
    model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
    messages=[
        {
            "role": "user", 
            "content": f"""
You are an oncology decision-support assistant.

SYNTHETIC PATIENT COHORT (Synthea/mCODE — not real patients):
{cohort_summary}

Question: {question}

Give a brief comparison. Always state data is synthetic.
"""
        }
    ],
    max_completion_tokens=300
)

print(f"❓ {question}")
print(f"💬 {response.choices[0].message.content}")

❓ Compare our patient counts across all 3 cancer types
💬 


In [19]:
# Debug — check what's actually coming back
question = "Compare our patient counts across all 3 cancer types"

cohort_data = get_cohort_stats()
cohort_summary = "\n".join([
    f"- {row[0]} cancer: {row[1]} patients ({row[2]} female, {row[3]} male)"
    for row in cohort_data
])

print("Cohort data being sent:")
print(cohort_summary)
print()

response = openai_client.chat.completions.create(
    model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
    messages=[
        {
            "role": "user",
            "content": f"Here is synthetic patient data:\n{cohort_summary}\n\nPlease summarize the patient counts for each cancer type in 2-3 sentences."
        }
    ],
    max_completion_tokens=300
)

print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Response: '{response.choices[0].message.content}'")
print(f"Usage: {response.usage}")

Cohort data being sent:
- breast cancer: 48 patients (42 female, 6 male)
- colorectal cancer: 28 patients (18 female, 10 male)
- lung cancer: 13 patients (9 female, 30 male)

Finish reason: length
Response: ''
Usage: CompletionUsage(completion_tokens=300, prompt_tokens=73, total_tokens=373, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=300, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 8, 'engine_ttft_ms': 34, 'engine_ttlt_ms': 2511, 'pre_inference_ms': 185, 'service_tbt_ms': 8, 'service_ttft_ms': 670, 'service_ttlt_ms': 3139, 'total_duration_ms': 2967, 'user_visible_ttft_ms': 485})


In [20]:
# Fix — increase tokens significantly for gpt-5-mini reasoning model
response = openai_client.chat.completions.create(
    model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
    messages=[
        {
            "role": "user",
            "content": f"""Here is synthetic patient data:
- breast cancer: 48 patients (42 female, 6 male)
- colorectal cancer: 28 patients (18 female, 10 male)
- lung cancer: 13 patients (9 female, 30 male)

In 2 sentences, summarize these patient counts. Note the data is synthetic."""
        }
    ],
    max_completion_tokens=2000
)

print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Response: {response.choices[0].message.content}")
print(f"Tokens used: {response.usage.completion_tokens}")

Finish reason: stop
Response: The dataset lists 48 breast cancer patients (42 female, 6 male), 28 colorectal cancer patients (18 female, 10 male), and 13 lung cancer patients — however, the provided lung cancer sex breakdown of 9 female and 30 male sums to 39 and is inconsistent with the stated total of 13. Note that these counts are synthetic.
Tokens used: 404


In [21]:
def answer_oncology_question(question):
    """
    Combined agent — SQL cohort data + gpt-5-mini reasoning
    """
    # Get cohort stats
    cohort_data = get_cohort_stats()
    cohort_summary = "\n".join([
        f"- {row[0]} cancer: {row[1]} patients ({row[2]} female, {row[3]} male)"
        for row in cohort_data
    ])

    # Get specific cancer details if mentioned
    extra_context = ""
    for cancer_type in ['breast', 'lung', 'colorectal']:
        if cancer_type in question.lower():
            age_data = get_age_distribution(cancer_type)
            age_summary = ", ".join([
                f"{row[0]}: {row[1]}" for row in age_data
            ])
            extra_context = f"\n{cancer_type.title()} age distribution: {age_summary}"

    # Build prompt
    prompt = f"""You are the Oncology Care Insights Agent — 
decision support for clinical ops analysts.

SYNTHETIC PATIENT COHORT (Synthea/mCODE — not real patients):
{cohort_summary}
{extra_context}

Always state data is synthetic in your answer.
Keep answers concise and clear.

Question: {question}"""

    response = openai_client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT'),
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000
    )

    return response.choices[0].message.content

# Final test — all 3 questions
print("=" * 60)
print("FINAL COMBINED AGENT TEST")
print("=" * 60)

questions = [
    "How many breast cancer patients do we have and what is their age breakdown?",
    "Compare our patient counts across all 3 cancer types",
    "How many lung cancer patients do we have?"
]

for q in questions:
    print(f"\n❓ {q}")
    answer = answer_oncology_question(q)
    print(f"💬 {answer}")
    print("-" * 60)

FINAL COMBINED AGENT TEST

❓ How many breast cancer patients do we have and what is their age breakdown?


OperationalError: (20004, b'DB-Lib error message 20004, severity 9:\nRead from the server failed\nNet-Lib error during Operation timed out (60)\n')

In [22]:
import pymssql
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv('../.env')

# Reconnect to Azure SQL
conn = pymssql.connect(
    server='oncology-sql-server.database.windows.net',
    user='oncologyadmin',
    password=os.getenv('AZURE_SQL_PASSWORD'),
    database='oncology-patients',
    port=1433
)
cursor = conn.cursor()
print("✅ Reconnected to Azure SQL!")

# Reconnect OpenAI client
openai_client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)
print("✅ Reconnected to OpenAI!")

# Quick test
cursor.execute("SELECT COUNT(*) FROM Patients")
count = cursor.fetchone()[0]
print(f"✅ Patients in database: {count}")

✅ Reconnected to Azure SQL!
✅ Reconnected to OpenAI!
✅ Patients in database: 95


In [23]:
# Final combined agent test
print("=" * 60)
print("FINAL COMBINED AGENT TEST")
print("=" * 60)

questions = [
    "How many breast cancer patients do we have and what is their age breakdown?",
    "Compare our patient counts across all 3 cancer types",
    "How many lung cancer patients do we have?"
]

for q in questions:
    print(f"\n❓ {q}")
    try:
        answer = answer_oncology_question(q)
        print(f"💬 {answer}")
    except Exception as e:
        # Reconnect if needed
        conn = pymssql.connect(
            server='oncology-sql-server.database.windows.net',
            user='oncologyadmin',
            password=os.getenv('AZURE_SQL_PASSWORD'),
            database='oncology-patients',
            port=1433
        )
        cursor = conn.cursor()
        answer = answer_oncology_question(q)
        print(f"💬 {answer}")
    print("-" * 60)

print("\n✅ Combined agent test complete!")

FINAL COMBINED AGENT TEST

❓ How many breast cancer patients do we have and what is their age breakdown?
💬 This is synthetic data.

Total breast cancer patients: 48 (42 female, 6 male).

Age breakdown (count, % of 48):
- Under 40: 19 (≈39.6%)
- 40–49: 4 (≈8.3%)
- 50–59: 5 (≈10.4%)
- 60–69: 8 (≈16.7%)
- 70+: 12 (25.0%)
------------------------------------------------------------

❓ Compare our patient counts across all 3 cancer types
💬 Note: these are synthetic (Synthea/mCODE) patients.

Summary (issue noted)
- Reported counts by cancer type: breast 48, colorectal 28, lung 13 → total = 89 patients.
- Reported gender breakdowns: breast 42F / 6M, colorectal 18F / 10M, lung 9F / 30M → gender sum = 115 (69F / 46M).

There is an inconsistency: lung cancer is listed as 13 patients but the lung gender counts sum to 39. Two ways to interpret:
1. If lung total = 13 (as written): cohort total = 89. Type distribution: breast 48/89 (54%), colorectal 28/89 (31.5%), lung 13/89 (14.6%). Gender totals 